# Part 4 · Errors, health checks and the demo UI

The last layer makes failures readable and the service observable from outside, then puts a UI on top.

| File | Role |
| --- | --- |
| [`core/errors.py`](../app/core/errors.py) | one error body for every failure (RFC 9457 Problem Details) |
| [`services/streaming.py`](../app/services/streaming.py) | failures **during** a stream become an `error` event; disconnect and timeout stop the run |
| [`routers/health.py`](../app/routers/health.py) | `/health` (liveness) and `/ready` (readiness) |
| [`static/`](../app/static/) | the demo chat UI, served by the same app |

Docs: [FastAPI handling errors](https://fastapi.tiangolo.com/tutorial/handling-errors/) ·
[RFC 9457](https://www.rfc-editor.org/rfc/rfc9457) ·
[Kubernetes probes](https://kubernetes.io/docs/concepts/workloads/pods/probes/) ·
[FastAPI static files](https://fastapi.tiangolo.com/tutorial/static-files/)

In [1]:
import logging

from dotenv import load_dotenv

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
for noisy in ("httpx", "httpx2"):  # HTTP clients log every request at INFO
    logging.getLogger(noisy).setLevel(logging.WARNING)
load_dotenv()

from utils import check_server, client, sse_events, show, short, source  # notebooks/utils.py

check_server()

server: ok


## 1. Errors before the stream: one body shape

`problem()` builds the RFC 9457 fields (`type`, `title`, `status`, `detail`, `instance`); the two handlers convert
every `HTTPException` (also FastAPI's own 404 for an unknown path) and every validation error (422) to it, with
`content-type: application/problem+json`. FastAPI has no built-in Problem Details, hence the handlers.

In [2]:
from app.core import errors

source(errors)

"""One error body for every failure: RFC 9457 Problem Details (`application/problem+json`)."""
from fastapi import FastAPI, Request
from fastapi.exceptions import RequestValidationError
from fastapi.responses import JSONResponse
from starlette.exceptions import HTTPException as StarletteHTTPException

TITLES = {401: "Unauthorized", 404: "Not Found", 409: "Conflict", 422: "Validation Error", 500: "Run Failed",
          503: "Service Unavailable", 504: "Run Timed Out"}


def problem(status: int, detail: str = "", instance: str = "") -> dict:
    return {"type": "about:blank", "title": TITLES.get(status, "Error"), "status": status, "detail": detail,
            "instance": instance}


async def http_error(request: Request, exc: StarletteHTTPException) -> JSONResponse:
    return JSONResponse(problem(exc.status_code, str(exc.detail), request.url.path), exc.status_code,
                        headers=exc.headers, media_type="application/problem+json")


async def validation_error(request: Request, exc: RequestValidationError) -> JSONResponse:
    body = problem(422, "The request body or parameters are invalid.", request.url.path)
    body["errors"] = [{"loc": e["loc"], "msg": e["msg"]} for e in exc.errors()]
    return JSONResponse(body, 422, media_type="application/problem+json")


def add_error_handlers(app: FastAPI) -> None:
    # Starlette's HTTPException also covers FastAPI's and the 404 of unknown routes
    app.add_exception_handler(StarletteHTTPException, http_error)
    app.add_exception_handler(RequestValidationError, validation_error)

In [3]:
alice = client("alice")
thread_id = alice.post("/threads").json()["thread_id"]
cases = [
    ("no token", lambda: client(None).get("/threads")),
    ("unknown thread", lambda: alice.get("/threads/000000000000")),
    ("resume a thread that is not waiting", lambda: alice.post(f"/threads/{thread_id}/resume",
                                                               json={"decision": "approve"})),
    ("empty message", lambda: alice.post(f"/threads/{thread_id}/runs", json={"message": ""})),
    ("unknown path", lambda: alice.get("/nothing-here")),
]
for label, call in cases:
    r = call()
    b = r.json()
    print(f"{label:<38}{r.status_code}  {r.headers['content-type']:<26}{b['title']}: {short(b['detail'], 45)}")
print("\n422 also lists the fields:", alice.post(f"/threads/{thread_id}/runs", json={"message": ""}).json()["errors"])

no token                              401  application/problem+json  Unauthorized: Missing or invalid bearer token
unknown thread                        404  application/problem+json  Not Found: Thread 000000000000 not found
resume a thread that is not waiting   409  application/problem+json  Conflict: Nothing to resume: the thread is idle
empty message                         422  application/problem+json  Validation Error: The request body or parameters are invalid.
unknown path                          404  application/problem+json  Not Found: Not Found

422 also lists the fields: [{'loc': ['body', 'message'], 'msg': 'String should have at least 1 character'}]


## 2. Errors during the stream

Once the first event is sent, the status `200` is gone: the server cannot switch to a 500. `run_events` catches the
exception, sends `event: error` with the same problem fields, and ends the stream. The detail never contains the
exception text (it can hold keys, paths, SQL); the server log keeps it under the run id.

To see it, call `run_events` with the real agent wrapped so that it fails after the first tool step, as a model
provider outage would. Then with a 3-second limit for the whole run: the run is stopped and reported as 504.

In [4]:
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import InMemorySaver

from app.agent.graph import build_agent
from app.core.config import get_settings
from app.services.streaming import run_events

agent = build_agent(init_chat_model(get_settings().model), InMemorySaver())


class FailsAfterFirstTool:
    """The real agent, but its stream raises after the first `tools` step (a provider outage in the middle)."""

    def __init__(self, agent):
        self.agent = agent

    async def astream(self, *args, **kwargs):
        async for mode, chunk in self.agent.astream(*args, **kwargs):
            yield mode, chunk
            if mode == "updates" and "tools" in chunk:
                raise ConnectionError("model provider unavailable")

    async def aget_state(self, config):
        return await self.agent.aget_state(config)


payload = {"messages": [{"role": "user", "content": "Brief on heat pump noise levels."}]}
async for e in run_events(FailsAfterFirstTool(agent), {}, "alice", "nb4-fail", payload):
    if e.event != "token":
        show(e.event, e.data)

INFO | run bf98c16c started: user=alice thread=nb4-fail


[metadata] run bf98c16c on thread nb4-fail
[step] model
[tool_call] internet_search({"query": "heat pump outdoor unit noise levels dB manufacturer specifications residential …)
[progress] searching: heat pump outdoor unit noise levels dB manufacturer specifications residential air source heat pump


INFO | internet_search('heat pump outdoor unit noise levels dB manufacturer specifications re…) -> ok 2664 ms
ERROR | run bf98c16c failed: ConnectionError('model provider unavailable')


[step] tools
[tool_result] internet_search -> success: [{"title": "Heat Pump Noise Requirements: Limits, Standards & Compliance Guide",…
[error] 500 Run Failed: The run stopped (ConnectionError); see the server log, run bf98c16c.


In [5]:
async for e in run_events(agent, {}, "alice", "nb4-timeout", payload, timeout_s=3):
    if e.event != "token":
        show(e.event, e.data)

INFO | run c91f7759 started: user=alice thread=nb4-timeout


[metadata] run c91f7759 on thread nb4-timeout
[step] model
[tool_call] internet_search({"query": "heat pump outdoor unit noise levels dB(A) typical manufacturer government"})
[progress] searching: heat pump outdoor unit noise levels dB(A) typical manufacturer government


WARNING | run c91f7759 timed out after 3 s


[error] 504 Run Timed Out: The run took longer than 3 s.


**The client goes away.** When the tab closes, the connection drops; the response task is cancelled, the
`CancelledError` reaches `agent.astream` and the run stops: no new model or tool call starts (a tool call already
running in a worker thread still finishes). The server logs `run … cancelled: client disconnected`; the steps saved
before stay in the thread.

In [6]:
cancel_thread = alice.post("/threads").json()["thread_id"]
with alice.stream("POST", f"/threads/{cancel_thread}/runs", json={"message": "Brief on induction cooktops."}) as r:
    for e in sse_events(r):
        print("received:", e["event"])
        if e["event"] == "tool_call":
            break  # leaving the with-block closes the connection
detail = alice.get(f"/threads/{cancel_thread}").json()
print("\nthread after the disconnect:", detail["status"], "| saved messages:", [m["role"] for m in detail["messages"]])

received: metadata
received: step
received: tool_call

thread after the disconnect: idle | saved messages: ['user', 'assistant']


## 3. Health checks: alive vs ready

- `GET /health` (**liveness**): the process answers. It checks nothing else, so a slow model provider never makes the
  service look dead and get restarted.
- `GET /ready` (**readiness**): the service can take a run now (the database answers). It never calls the LLM: a probe
  must be fast and free. Neither needs a token: a load balancer or an orchestrator calls them.

In [7]:
from app.routers import health

source(health)

"""Liveness (/health: the process answers) and readiness (/ready: it can serve a run now). No token: a load balancer
or an orchestrator calls them. Neither calls the LLM."""
import logging

from fastapi import APIRouter, HTTPException

from app.dependencies import Store

router = APIRouter(tags=["health"])
logger = logging.getLogger(__name__)


@router.get("/health")
async def health() -> dict:
    return {"status": "ok"}


@router.get("/ready")
async def ready(store: Store) -> dict:
    try:
        await store.conn.execute("SELECT 1")  # the database answers
    except Exception as exc:
        logger.error("readiness check failed: %r", exc)
        raise HTTPException(503, "storage not ready")  # details stay in the log
    return {"status": "ready", "checks": {"store": "ok"}}

In [8]:
for path in ("/health", "/ready"):
    r = client(None).get(path)
    print(f"{path:<8}{r.status_code}  {r.json()}")

/health 200  {'status': 'ok'}
/ready  200  {'status': 'ready', 'checks': {'store': 'ok'}}


## 4. The demo UI

`main.py` serves `app/static/` and returns `index.html` at `/`. The page (`app.js`) reads each run with `fetch()` and
`response.body`, not with the browser's `EventSource`: `EventSource` only opens a GET request and cannot send an
`Authorization` header or a JSON body.

Open the service root `/` in a browser (`http://127.0.0.1:8000/` with the start command of Part 1):
1. Sign in with Alice's token (`alice-token-7f3a`, from `data/users.json`).
2. Ask for a brief. Open the activity block under the question to watch `tool_call`, `progress`, `tool_result`
   arrive while the agent works.
3. The approval card shows the brief `save_brief` would write: **Approve** (or **Reject** with a reason). The summary
   then streams in token by token.
4. Sign out and sign in as Bob (`bob-token-91c2`): his list has none of Alice's briefs.

<img src="images/ui_approval.png" width="640" alt="the demo UI: activity block and approval card">

## Optional extension (~10 min): trace each API run in Langfuse

Behind an API the Langfuse callback goes into the run config, and the user and thread become trace attributes, so
one user's runs can be filtered (keys as in Session 07).

1. `uv add langfuse==5.0.2`, and add `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY`, `LANGFUSE_BASE_URL` to `.env`.
2. In `app/services/streaming.py` (`run_events`, after `config = {...}`) and `app/routers/runs.py`:
   ```python
   from langfuse import propagate_attributes
   from langfuse.langchain import CallbackHandler

   config["callbacks"] = [CallbackHandler()]  # one trace per run

   # in create_run() and resume_run()
   with propagate_attributes(user_id=user.user_id, session_id=thread.thread_id):
       async for event in run_events(...):
           yield event
   ```
3. Restart the service, run a brief from the UI, open **Tracing** in Langfuse and filter by user `alice`.